# Data Transformation
receive a pdf, return a chroma data folder which is an embedding for the text, images and tables of the pdf.

In [1]:
import unstructured
from unstructured.partition.pdf import partition_pdf
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

from dotenv import load_dotenv
import os

load_dotenv()  
import time

In [ ]:
embed_model = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")

import warnings
warnings.filterwarnings(
    "ignore",
    message="The `max_size` parameter is deprecated*"
)


file = input("Enter file directory: ")

raw_data = partition_pdf(
    filename=file,
    strategy="hi_res",
    infer_table_structure=True,
    extract_images_in_pdf=True,
    #chunking_strategy="by_title",
    extract_image_block_output_dir="./data/temp_images",
    languages=["eng", "fra"]
)


In [3]:
file = os.getcwd() + "/" + file
file

'/home/LouiZz/Documents/ragbot/cc.pdf'

In [4]:
print(list(raw_data))

[<unstructured.documents.elements.Title object at 0x7f4cb1618050>, <unstructured.documents.elements.NarrativeText object at 0x7f4cb16374a0>, <unstructured.documents.elements.NarrativeText object at 0x7f4cbb7bf850>, <unstructured.documents.elements.NarrativeText object at 0x7f4c701ae7d0>, <unstructured.documents.elements.NarrativeText object at 0x7f4cb1618a60>, <unstructured.documents.elements.Table object at 0x7f4c7012e7b0>, <unstructured.documents.elements.Header object at 0x7f4c9546b110>, <unstructured.documents.elements.Header object at 0x7f4c9546ad50>, <unstructured.documents.elements.Header object at 0x7f4c68713770>, <unstructured.documents.elements.Title object at 0x7f4cb1586ba0>, <unstructured.documents.elements.NarrativeText object at 0x7f4cb1587bd0>, <unstructured.documents.elements.NarrativeText object at 0x7f4cb15879a0>, <unstructured.documents.elements.Title object at 0x7f4cb1587d90>, <unstructured.documents.elements.Title object at 0x7f4c70102f90>, <unstructured.documents.

## Divide text, images and tables

In [5]:
text_elements = []
table_elements = []
image_elements = []

text_metadata = []

for element in raw_data:
    el_type = str(type(element))

    if "Table" in el_type:
        table_elements.append(element)
    elif "Image" in el_type:
        image_elements.append(element)
    elif "CompositeElement" in el_type or "Text" in el_type:
        text_elements.append(element)
        text_metadata.append({"page_number":element.metadata.page_number,
                              "source": file,
                              "file_name": element.metadata.filename,
                              "type": "text"})
        


In [6]:
print(text_elements)

[<unstructured.documents.elements.NarrativeText object at 0x7f4cb16374a0>, <unstructured.documents.elements.NarrativeText object at 0x7f4cbb7bf850>, <unstructured.documents.elements.NarrativeText object at 0x7f4c701ae7d0>, <unstructured.documents.elements.NarrativeText object at 0x7f4cb1618a60>, <unstructured.documents.elements.NarrativeText object at 0x7f4cb1587bd0>, <unstructured.documents.elements.NarrativeText object at 0x7f4cb15879a0>, <unstructured.documents.elements.NarrativeText object at 0x7f4c70102e40>, <unstructured.documents.elements.Text object at 0x7f4c701030e0>, <unstructured.documents.elements.NarrativeText object at 0x7f4c70103380>, <unstructured.documents.elements.NarrativeText object at 0x7f4c70103540>, <unstructured.documents.elements.NarrativeText object at 0x7f4c701037e0>, <unstructured.documents.elements.Text object at 0x7f4cb1587b60>, <unstructured.documents.elements.NarrativeText object at 0x7f4c70102c80>, <unstructured.documents.elements.NarrativeText object a

In [7]:
raw_text = "\n\n".join(str(el) for el in text_elements)
print(raw_text)

Localisation et Cartographie Simultanées (EKF & RTS Smoother)

Mohammad SWAYDAN ET Hassan HUSSEIN DIT SAFADI ENSTA

5 janvier 2026

Table des matières

Ce Bureau d’Études porte sur la navigation autonome du robot sous-marin Redermor lors d’une mission de deux heures dans la baie de Douarnenez. Le robot est équipé de capteurs pro- prioceptifs (centrale inertielle, loch Doppler, capteur de pression) et d’un sonar latéral permettant de détecter des amers (mines).

L’objectif est double : estimer la trajectoire du robot malgré la dérive des capteurs (Navigation à l’estime) puis corriger cette trajectoire et cartographier l’environnement en utilisant un Filtre de Kalman Étendu (EKF) et un Lisseur de Rauch-Tung-Striebel (RTS).

Le vecteur d’état du robot est sa position p = [x,y,z]T. L’évolution cinématique est donnée par ˙p(t) = R(φ,θ,ψ) · vr(t). Nous avons implémenté une intégration numérique par la méthode d’Euler (dt = 0.1s) :

Figure 1 – Trajectoire calculée par estime. La dérive tempor

## Text Chunking

In [8]:
semantic_chunker = SemanticChunker(embeddings=embed_model, breakpoint_threshold_amount=82, breakpoint_threshold_type="percentile")
print(semantic_chunker)

In [9]:
text_chunks = semantic_chunker.create_documents([raw_text], metadatas=text_metadata)
print(len(text_chunks))

7


In [10]:
print(len(text_elements)), print(len(image_elements)), print(len(table_elements))

31
4
2


(None, None, None)

In [11]:
text_chunks[0].page_content

'Localisation et Cartographie Simultanées (EKF & RTS Smoother)\n\nMohammad SWAYDAN ET Hassan HUSSEIN DIT SAFADI ENSTA\n\n5 janvier 2026\n\nTable des matières\n\nCe Bureau d’Études porte sur la navigation autonome du robot sous-marin Redermor lors d’une mission de deux heures dans la baie de Douarnenez. Le robot est équipé de capteurs pro- prioceptifs (centrale inertielle, loch Doppler, capteur de pression) et d’un sonar latéral permettant de détecter des amers (mines). L’objectif est double : estimer la trajectoire du robot malgré la dérive des capteurs (Navigation à l’estime) puis corriger cette trajectoire et cartographier l’environnement en utilisant un Filtre de Kalman Étendu (EKF) et un Lisseur de Rauch-Tung-Striebel (RTS). Le vecteur d’état du robot est sa position p = [x,y,z]T. L’évolution cinématique est donnée par ˙p(t) = R(φ,θ,ψ) · vr(t).'

In [12]:
type(text_chunks[0])

langchain_core.documents.base.Document

In [13]:
text_chunks[0].metadata

{'page_number': 1,
 'source': '/home/LouiZz/Documents/ragbot/cc.pdf',
 'file_name': 'cc.pdf',
 'type': 'text'}

In [14]:
for i in text_chunks:
    print(i)

page_content='Localisation et Cartographie Simultanées (EKF & RTS Smoother)

Mohammad SWAYDAN ET Hassan HUSSEIN DIT SAFADI ENSTA

5 janvier 2026

Table des matières

Ce Bureau d’Études porte sur la navigation autonome du robot sous-marin Redermor lors d’une mission de deux heures dans la baie de Douarnenez. Le robot est équipé de capteurs pro- prioceptifs (centrale inertielle, loch Doppler, capteur de pression) et d’un sonar latéral permettant de détecter des amers (mines). L’objectif est double : estimer la trajectoire du robot malgré la dérive des capteurs (Navigation à l’estime) puis corriger cette trajectoire et cartographier l’environnement en utilisant un Filtre de Kalman Étendu (EKF) et un Lisseur de Rauch-Tung-Striebel (RTS). Le vecteur d’état du robot est sa position p = [x,y,z]T. L’évolution cinématique est donnée par ˙p(t) = R(φ,θ,ψ) · vr(t).' metadata={'page_number': 1, 'source': '/home/LouiZz/Documents/ragbot/cc.pdf', 'file_name': 'cc.pdf', 'type': 'text'}
page_content='No

In [15]:
type(table_elements[0])

unstructured.documents.elements.Table

In [16]:
table_elements[0].text

'1 Introduction 2 Modélisation et Navigation à l’Estime (Q1 & Q2) 2.1 Modèle d’État et Intégration . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 2.2 Démonstration de la Covariance du Bruit (Q2) . . . . . . . . . . . . . . . . . . . 3 Prédiction de l’Incertitude (Q3) 4 SLAM : Filtrage de Kalman Étendu (Q4 & Q5) 4.1 Modèle d’Observation (Q4) . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4.2 Résultats EKF (Q5) . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 5 Lissage de Rauch-Tung-Striebel (Q6) 5.1 Analyse de la précision finale . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 6 Conclusion A Annexe : Codes Python (Extraits) A.1 Algorithme EKF (Mise à jour) . . . . . . . . . . . . . . . . . . . . . . . . . . . . A.2 Algorithme Lisseur RTS (Backward) . . . . . . . . . . . . . . . . . . . . . . . . . 2 2 2 2 2 3 3 3 4 4 5 6 6 6'

## Tables

In [17]:
table_docs = []
for table in table_elements:
    page_content = table.text
    metadata = {
        "page_number":table.metadata.page_number,
        "type":"table",
        "html_content":table.metadata.text_as_html
                }
    
    document = Document(page_content=page_content, metadata=metadata)
    table_docs.append(document)

In [18]:
(image_elements[0].metadata.image_path)

'./temp_images/figure-2-1.jpg'

In [19]:
type(table_docs[0])

langchain_core.documents.base.Document

In [20]:
table_docs[0].metadata

{'page_number': 1,
 'type': 'table',
 'html_content': "<table><thead><tr><th>1 2</th><th>Introduction Modélisation et Navigation a</th><th></th><th>I'Estime</th><th></th><th>(Q1</th><th>&amp;</th><th></th><th>Q2)</th><th></th><th></th></tr></thead><tbody><tr><td></td><td>2.1 Modéle d’Etat et Intégration|</td><td></td><td></td><td></td><td></td><td></td><td></td><td></td><td></td><td></td></tr><tr><td></td><td>2.2 Démonstration de la Covariance</td><td></td><td></td><td>Bruit</td><td></td><td>(Q2</td><td></td><td></td><td></td><td></td></tr><tr><td>3</td><td>Prédiction de I'Incertitude</td><td>(Q3)| Etendu</td><td></td><td></td><td></td><td></td><td></td><td></td><td></td><td></td></tr><tr><td>6</td><td>Lissage de Rauch-Tung-Striebel Analyse de la précision finale Conclusion|</td><td></td><td>(Q6)</td><td></td><td></td><td></td><td></td><td></td><td></td><td></td></tr><tr><td>A</td><td>Annexe : Codes Python (Extraits)</td><td></td><td></td><td></td><td></td><td></td><td></td><td></td><t

## Images

In [21]:
from langchain_groq import ChatGroq
import base64
from langchain_core.messages import HumanMessage

llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct",
               temperature=0.1,
               max_tokens=500) # for a small paragraph

In [22]:
def describe_image_with_LLM(path: str)->str:
    """
    This function will return a description of the image as a string format using a multimodal LM
    
    Args
        path: str, path of file

    Return
        description: str
    """

    prompt = """
    You are an expert technical assistant, analyze this image from an engineering document.
    1. Identify the type (Diagram, Plot, Circuit, or Photo)
    2. Transcribe any visible text, lables, equations, or axis values
    3. Describe the structural relationships or trends shown
    output a concise, dense paragraph optimized for retrieval.    
    """

    with open(path, "rb") as f:
        image_bytes = f.read()

    b64_string = base64.b64encode(image_bytes).decode("utf-8")


    message = HumanMessage( content=[ {"type": "text", 
                                       "text": prompt},
                                         { "type": "image_url",
                                           "image_url": {"url": f"data:image/jpeg;base64,{b64_string}"} }, ] )
    
    return llm.invoke([message])


In [23]:
describe_image_with_LLM(image_elements[0].metadata.image_path)


AIMessage(content='The image presents two diagrams: a 3D trajectory plot on the left and a 2D top-view plot on the right. \n**Type:** Diagrams\n**Transcribed text and labels:**\n- Left diagram: \n  - "Trajectoire 3D (Q1)"\n  - Axes labels: \n    - X (m): 0 to 800\n    - Y (m): -200 to 800\n    - Z (m): 0 to 20\n- Right diagram: \n  - "Vue de dessus (XY)"\n  - Axes labels: \n    - X (m): -200 to 1000\n    - Y (m): -200 to 600\n**Structural relationships or trends:** \nThe 3D trajectory plot shows a complex path with varying elevations, while the 2D top-view plot displays the same path from above, revealing a dense network of interconnected lines. Both plots appear to represent the movement or trajectory of an object within a defined space, with the 3D plot providing a more comprehensive view of the path\'s spatial characteristics and the 2D plot offering a clearer understanding of the path\'s planar structure.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens

In [24]:

image_docs = []
for image in image_elements:
    desc = describe_image_with_LLM(image.metadata.image_path)
    
    page_content = str(desc)
    metadata = {
        "model":"meta-llama/llama-4-scout-17b-16e-instruct",
        "source": file,
        "type": "image",
        "page": image.metadata.page_number,
        "original_path":image.metadata.image_path
    }
    document = Document(metadata=metadata, page_content=page_content)
    image_docs.append(document)

    time.sleep(2)

In [25]:
image_docs[0]

Document(metadata={'model': 'meta-llama/llama-4-scout-17b-16e-instruct', 'source': '/home/LouiZz/Documents/ragbot/cc.pdf', 'type': 'image', 'page': 2, 'original_path': './temp_images/figure-2-1.jpg'}, page_content='content=\'The image presents two diagrams: a 3D trajectory plot on the left and a 2D top-down view on the right. **Type:** Diagrams. Visible text includes "Trajectoire 3D (Q1)" and "Vue de dessus (XY)". Axis labels are X (m), Y (m), and Z (m) with ranges from 0 to 800 for X, -200 to 600 for Y, and 0 to 20 for Z. The 3D plot shows a trajectory with varying Z heights across the X-Y plane, indicating changes in altitude over a 3D path. The 2D plot displays the same trajectory from a top-down perspective, highlighting movements within the X-Y plane. **Structural relationships:** The trajectory shows a complex path with multiple turns and altitude changes, suggesting a dynamic movement pattern, possibly from a drone or vehicle.\' additional_kwargs={} response_metadata={\'token_us

In [26]:
type(image_docs[0])

langchain_core.documents.base.Document

# Implement RAG

add all documents into one list

In [27]:
documents = []

for i in text_chunks:
    documents.append(i)

for i in image_docs:
    documents.append(i)

for i in table_docs:
    documents.append(i)


In [28]:
documents[0]

Document(metadata={'page_number': 1, 'source': '/home/LouiZz/Documents/ragbot/cc.pdf', 'file_name': 'cc.pdf', 'type': 'text'}, page_content='Localisation et Cartographie Simultanées (EKF & RTS Smoother)\n\nMohammad SWAYDAN ET Hassan HUSSEIN DIT SAFADI ENSTA\n\n5 janvier 2026\n\nTable des matières\n\nCe Bureau d’Études porte sur la navigation autonome du robot sous-marin Redermor lors d’une mission de deux heures dans la baie de Douarnenez. Le robot est équipé de capteurs pro- prioceptifs (centrale inertielle, loch Doppler, capteur de pression) et d’un sonar latéral permettant de détecter des amers (mines). L’objectif est double : estimer la trajectoire du robot malgré la dérive des capteurs (Navigation à l’estime) puis corriger cette trajectoire et cartographier l’environnement en utilisant un Filtre de Kalman Étendu (EKF) et un Lisseur de Rauch-Tung-Striebel (RTS). Le vecteur d’état du robot est sa position p = [x,y,z]T. L’évolution cinématique est donnée par ˙p(t) = R(φ,θ,ψ) · vr(t).

In [ ]:
from langchain_chroma import Chroma
DB_PATH = "./data/chroma_db_data" # Where the database will be saved on disk
vectorstore = Chroma.from_documents(documents=documents, embedding=embed_model, persist_directory=DB_PATH )

In [32]:
type(vectorstore)

langchain_chroma.vectorstores.Chroma